# Notebook 06 — Clustering: Leiden and Louvain

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 06 of 12  
**Time estimate:** 75 minutes

> Clustering assigns a label to each cell so that cells in the same cluster are
> more similar to each other than to cells in other clusters.
> In single-cell analysis, graph-based clustering on a kNN graph is the
> dominant approach — not k-means or hierarchical clustering.

---
## Step 1 — Motivation

After QC, normalization, PCA, and UMAP, we need to *assign cell types* in a
principled, reproducible way. Manual gating is subjective and doesn't scale to
tens of thousands of cells. Graph-based clustering (Leiden/Louvain) provides
unsupervised, reproducible cluster assignments.

---
## Step 2 — Intuition

**Step 1 — Build a kNN graph:** For each cell, find its k nearest neighbours in PCA
space. Connect each cell to its neighbours. The result is a weighted graph where
edges connect similar cells.

**Step 2 — Community detection (Leiden/Louvain):** Partition the graph into
"communities" (clusters) that have more edges within communities than between them.
The `resolution` parameter controls granularity:
- Low resolution → fewer, larger clusters
- High resolution → more, smaller clusters

**Leiden vs. Louvain:**  
Louvain can produce internally disconnected communities (a bug). Leiden (Traag 2019)
fixes this — all Leiden communities are well-connected. Leiden is now the default;
Louvain is legacy.

---
## Step 3 — Biological Background

**Why kNN-graph-based clustering?**  
Single-cell data forms manifolds in high-dimensional space, not globular clusters.
k-means assumes spherical clusters of equal size — a poor match. Graph-based methods
make no assumptions about cluster shape and scale to hundreds of thousands of cells.

**Choosing resolution:**  
A resolution of 0.1–0.3 typically recovers major cell types (T cells, B cells,
monocytes). A resolution of 1.0–2.0 may split T cells into CD4+ and CD8+
subtypes. Resolution should be guided by biology — what granularity of cell type
classification does your question require?

**Shared Nearest Neighbours (SNN):**  
The kNN graph is often refined to a Shared Nearest Neighbour (SNN) graph: edge weight
= number of shared neighbours / (k + k - shared neighbours). SNN downweights spurious
edges in noisy regions of the embedding.

---
## Step 4 — Mathematical Explanation

**kNN graph construction:**  
For each cell $i$, compute distance to all other cells in PCA space, take the $k$
smallest. Adjacency matrix $A$ has $A_{ij} = 1$ if $j \in \text{kNN}(i)$ or
$i \in \text{kNN}(j)$.

**Modularity (Louvain objective):**
$$Q = \frac{1}{2m} \sum_{ij} \left[A_{ij} - \frac{k_i k_j}{2m}\right] \delta(c_i, c_j)$$

where $m$ = total edge weight, $k_i$ = degree of node $i$, $c_i$ = cluster of cell $i$,
$\delta(c_i, c_j) = 1$ if cells are in the same cluster.

**Resolution parameter:** A resolution $\gamma > 1$ makes the algorithm prefer
smaller clusters; $\gamma < 1$ prefers larger. In practice, resolution is treated
as a tuning parameter.

In [ ]:
# Step 6 — Python: kNN graph from scratch + Leiden via leidenalg library
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

rng = np.random.default_rng(42)

# ---- Simulate PCA-reduced single-cell data ----
N_CELLS = 300
cell_types_true = rng.choice(['T_cell', 'B_cell', 'Monocyte'], N_CELLS, p=[0.4, 0.35, 0.25])
ct_colors_true = {'T_cell': 'steelblue', 'B_cell': 'tomato', 'Monocyte': 'forestgreen'}

# Simulate 30-dim PCA coordinates with known cluster structure
centers = {'T_cell': np.array([3.0, 0.0] + [0.0]*28),
            'B_cell': np.array([-2.5, 2.0] + [0.0]*28),
            'Monocyte': np.array([0.0, -3.5] + [0.0]*28)}
Z_pca = np.vstack([
    centers[ct] + rng.normal(0, 0.8, 30) for ct in cell_types_true
])

# ---- Build kNN graph from scratch ----
def build_knn_graph(Z, k=15):
    """Build symmetric kNN adjacency matrix."""
    n = Z.shape[0]
    # Euclidean distances
    D = cdist(Z, Z, metric='euclidean')
    np.fill_diagonal(D, np.inf)  # ignore self-distance
    # For each cell, keep k nearest neighbours
    knn_idx = np.argsort(D, axis=1)[:, :k]
    # Build symmetric adjacency
    A = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in knn_idx[i]:
            A[i, j] = 1.0
            A[j, i] = 1.0  # symmetrize
    return A, knn_idx

A, knn_idx = build_knn_graph(Z_pca, k=15)
print(f'kNN graph: {A.shape}, {int(A.sum()//2)} undirected edges')

# ---- Leiden clustering via leidenalg ----
try:
    import igraph as ig
    import leidenalg
    # Build igraph from adjacency
    sources, targets = np.where(np.triu(A, k=1) > 0)
    g = ig.Graph(n=N_CELLS, edges=list(zip(sources.tolist(), targets.tolist())))

    labels_leiden = {}
    for res in [0.3, 0.6, 1.0]:
        partition = leidenalg.find_partition(
            g,
            leidenalg.RBConfigurationVertexPartition,
            resolution_parameter=res,
            seed=42
        )
        labels_leiden[res] = np.array(partition.membership)
        print(f'Resolution {res:.1f}: {len(set(partition.membership))} clusters')
    leiden_available = True
except ImportError:
    print('leidenalg / igraph not installed. Using a simple greedy label propagation fallback.')
    leiden_available = False
    # Simple label propagation as a fallback
    def greedy_cluster(A, n_steps=5, rng=None):
        """Trivial label propagation — not Leiden, just a fallback."""
        labels = np.arange(A.shape[0])
        for _ in range(n_steps):
            for i in rng.permutation(A.shape[0]):
                nbrs = np.where(A[i] > 0)[0]
                if len(nbrs) > 0:
                    nbr_labels = labels[nbrs]
                    labels[i] = np.bincount(nbr_labels).argmax()
        # Re-index
        uniq = {v: k for k, v in enumerate(sorted(set(labels.tolist())))}
        return np.array([uniq[l] for l in labels])
    lp_labels = greedy_cluster(A, rng=rng)
    labels_leiden = {0.3: lp_labels, 0.6: lp_labels, 1.0: lp_labels}
    print(f'Fallback: {len(set(lp_labels.tolist()))} clusters')

In [ ]:
# Step 7 — Visualization: kNN graph, clusters at different resolutions
# Use PCA 2D for display (or a simple 2D projection)
from sklearn.decomposition import PCA as skPCA
pca2 = skPCA(n_components=2)
try:
    from sklearn.decomposition import PCA as skPCA
    Z2 = skPCA(n_components=2).fit_transform(Z_pca)
except:
    Z2 = Z_pca[:, :2]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Panel A: True cell types
ax = axes[0]
for ct, col in ct_colors_true.items():
    mask = np.array(cell_types_true) == ct
    ax.scatter(Z2[mask, 0], Z2[mask, 1], c=col, s=12, alpha=0.7, label=ct)
ax.set_title('A. Ground truth cell types')
ax.legend(fontsize=7)

# Panels B-D: Leiden at 3 resolutions
cmap = plt.cm.tab10
for j, res in enumerate([0.3, 0.6, 1.0]):
    ax = axes[j + 1]
    labels = labels_leiden[res]
    n_clusters = len(set(labels.tolist()))
    for cl in sorted(set(labels.tolist())):
        mask = labels == cl
        ax.scatter(Z2[mask, 0], Z2[mask, 1],
                    c=[cmap(cl / max(n_clusters, 1))], s=12, alpha=0.7, label=f'C{cl}')
    ax.set_title(f'{chr(66+j)}. Leiden resolution={res}\n({n_clusters} clusters)')
    if n_clusters <= 6:
        ax.legend(fontsize=6)

plt.suptitle('Module 10 NB06: Graph-Based Clustering (Leiden)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scrna_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement `build_knn_graph(Z, k)` from scratch using only numpy and scipy.
2. Run Leiden at resolutions 0.1, 0.5, and 2.0. What is the minimum and maximum
   number of clusters found?
3. Compute the Adjusted Rand Index (ARI) between Leiden clusters and the true cell
   type labels at each resolution. Which resolution gives the best match?
4. What does it mean biologically if Leiden at resolution 1.0 splits what was one
   cluster at resolution 0.3?

---
## Step 10 — Quiz

1. What is the input to Leiden clustering and what does it output?
2. Why is graph-based clustering preferred over k-means for scRNA-seq?
3. What is the `resolution` parameter and what does increasing it do?
4. What is the key difference between Leiden and Louvain?
5. In scanpy: what step must precede `sc.tl.leiden(adata)`?

---
## Step 12 — Reflection

> *[In one paragraph: explain why choosing a clustering resolution is ultimately
> a biological question, not a computational one, and describe two strategies a
> researcher could use to choose an appropriate resolution.]*

---
*Next: `07_marker_genes_and_cell_type_annotation.ipynb`*